### **UTILIZATION OF CNN3D ON MNIST DATASET**

In [ ]:
# imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as snshy5h5y
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.inage import ImageDataGenerator
from sklearn.model_selection import train_test_split

ImportError: Traceback (most recent call last):
  File "c:\Users\Axel John Nuqui\AppData\Local\Programs\Python\Python310\lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
# loading the dataset

train_path = pd.read_csv("dataset/mnist_train.csv")
test_path = pd.read_csv("dataset/mnist_test.csv")

print("Dataset Loaded Successfully!")

# showing the shapes if each csv and showing the first 5 records
print(f"Train Shape: {train_path.shape}")
print(f"Test Shape: {test_path.shape}")
print(f"First 5 records of the train dataset: \n {train_path.head()}")

# showing the columns for both train and test csvs
print(f"Train CSV Columns: {train_path.columns.tolist()}")
print(f"Test CSV Columns: {test_path.columns.tolist()}")

In [ ]:
# preprocessing train set
X = train_path.iloc[:, 1:].values / 255.0
y = train_path.iloc[:, 0].values

X = X.reshape(-1, 28, 28, 1)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

y_train = to_categorical(y_train, 10)
y_val   = to_categorical(y_val, 10)

print(f"Original X shape: {X.shape}")
print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")


# preprocessing test set - dropping the label column
X_test = test_path.iloc[:, 1:].values / 255.0
X_test = X_test.reshape(-1, 28, 28, 1)
print(f"X_test: {X_test.shape}")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# building the cnn3d model
def build_cnn3d_model():
    model = Sequential([
        Conv3D(32,(3,3,1),activation='relu',input_shape=(28,28,1,1)),
        MaxPooling3D((2,2,1)),
        Conv3D(64,(3,3,1),activation='relu'),
        Flatten(),
        Dense(128,activation='relu'),
        Dropout(0.5),
        Dense(10,activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
# === NO AUGMENTATION MODEL ===

print("Training started for MODEL with NO AUGMENTATION!")
no_aug_model = build_cnn3d_model()
no_aug_history = no_aug_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

# === AUGMENTATION MODEL ===

print("Training started for MODEL with AUGMENTATION!")
datagen = ImageDataGenerator(rotation_range=10,
                             width_shift_range=0.1,
                             height_shift_range=0.1,
                             zoom_range=0.1)

# reshape for augmentation
X_train_aug = X_train.reshape(-1,28,28,1)
datagen.fit(X_train_aug)

def generator():
    for X_batch, y_batch in datagen.flow(X_train_aug, y_train, batch_size=128):
        yield X_batch.reshape(-1,28,28,1,1), y_batch

augmented_model = build_cnn3d_model()
augmented_history = augmented_model.fit(
    generator(),
    steps_per_epoch=len(X_train)//128,
    epochs=10,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
# reshape for augmentation
X_train_aug = X_train.reshape(-1,28,28,1)
datagen.fit(X_train_aug)

def generator():
    for X_batch, y_batch in datagen.flow(X_train_aug, y_train, batch_size=128):
        yield X_batch.reshape(-1,28,28,1,1), y_batch

augmented_model = build_cnn3d_model()
augmented_history = augmented_model.fit(
    generator(),
    steps_per_epoch=len(X_train)//128,
    epochs=10,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
plt.figure(figsize=(12,10))

# Augmented Accuracy
plt.subplot(2,2,1)
plt.plot(augmented_history.history['accuracy'], label='Train Accuracy')
plt.plot(augmented_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model with Augmentation - Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Augmented Loss
plt.subplot(2,2,2)
plt.plot(augmented_history.history['loss'], label='Train Loss')
plt.plot(augmented_history.history['val_loss'], label='Validation Loss')
plt.title('Model with Augmentation - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# No Aug Accuracy
plt.subplot(2,2,3)
plt.plot(no_aug_history.history['accuracy'], label='Train Accuracy')
plt.plot(no_aug_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model with No Augmentation - Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# No Aug Loss
plt.subplot(2,2,4)
plt.plot(no_aug_history.history['loss'], label='Train Loss')
plt.plot(no_aug_history.history['val_loss'], label='Validation Loss')
plt.title('Model with No Augmentation - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# test prediction using cnn 3d model

preds = augmented_model.predict(X_test)
y_pred = np.argmax(preds, axis=1)

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(X_test)+1),
    "Label": y_pred
})

submission.to_csv("cnn3d_submission.csv", index=False)
print(submission.head())